In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from linearmodels import IV2SLS
from scipy import stats

# Problem 4 - Monte Carlo simulation

In [8]:
def simulate(gamma0,gamma1,gamma2):
    #definer parameter værdier
    n = 500 
    beta0 = 0.5
    beta1 = -0.1
    z = np.random.normal(loc = 0, scale = 1, size = n)
    xi = np.random.normal(loc = 0, scale = 1, size = n)
    v = np.random.normal(loc = 0, scale = 1, size = n)
    eta0 = (gamma0 - beta0) / (beta1 - gamma1)
    eta1 = gamma2 / (beta1 - gamma1)
    u = (v - xi) / (beta1 - gamma1)
    p = eta0 + eta1*z + u 
    y = beta0 + beta1*p + xi

    # Opretter dataframe med estimeret værdier
    df = pd.DataFrame({
        'y': y,
        'z': z,
        'p': p
    })

    # Estimere modellen med IV2SLS, z bruges som instrument for p 

    result = IV2SLS.from_formula('y ~ 1 + [p ~ z]', data = df).fit()
    beta1 = result.params['p']
    return beta1

def monte_carlo(reps=500):
    np.random.seed(0)
    IV_result_S1 = []
    IV_result_S2 = []
    IV_result_S3 = []
    for r in range(reps):
        beta1_S1 = simulate(gamma0=0.25,gamma1=1,gamma2=1) # S1
        beta1_S2 = simulate(gamma0=0.25,gamma1=1,gamma2=0.25) # S2
        beta1_S3 = simulate(gamma0=0.25,gamma1=0.25,gamma2=1) # S3
        IV_result_S1.append(beta1_S1)
        IV_result_S2.append(beta1_S2)
        IV_result_S3.append(beta1_S3)

    results = pd.DataFrame({'beta1_S1': IV_result_S1, 
                            'beta1_S2': IV_result_S2,
                            'beta1_S3': IV_result_S3
                            })

    return results


results1 = monte_carlo()
print(results1.describe())

         beta1_S1    beta1_S2    beta1_S3
count  500.000000  500.000000  500.000000
mean    -0.100081   -0.153499   -0.100063
std      0.048940    0.236936    0.016037
min     -0.271320   -1.302636   -0.156521
25%     -0.130896   -0.254323   -0.109999
50%     -0.095669   -0.118825   -0.099120
75%     -0.066584    0.003103   -0.088348
max      0.039020    0.453475   -0.062505
